In [1]:
!git clone https://github.com/Felix-Schwer/rainfall_kf.git
%cd /content/rainfall_kf
#!git pull origin main
#!pip uninstall -y rainfall_kf
!pip install -e .

Cloning into 'rainfall_kf'...
remote: Enumerating objects: 145, done.
remote: Total 145 (delta 0), reused 0 (delta 0), pack-reused 145 (from 1)
Receiving objects: 100% (145/145), 99.31 MiB | 44.33 MiB/s, done.
Resolving deltas: 100% (36/36), done.
/content/rainfall_kf
Obtaining file:///content/rainfall_kf
  Preparing metadata (setup.py) ... done
  Running setup.py develop for rainfall_kf


In [2]:
import xarray as xr

In [3]:
da = xr.open_dataset('/content/rainfall_kf/data/gpm_sjv_huc8_subset.nc')
db = xr.open_dataset('/content/rainfall_kf/data/gpm_sjv_subset.nc')

In [4]:
da

<xarray.Dataset> Size: 27MB
Dimensions:        (time: 7671, lat: 26, lon: 34)
Coordinates:
  * time           (time) datetime64[ns] 61kB 2000-01-01 ... 2020-12-31
  * lat            (lat) float32 104B 36.35 36.45 36.55 ... 38.65 38.75 38.85
  * lon            (lon) float32 136B -121.9 -121.8 -121.8 ... -118.8 -118.7
Data variables:
    precipitation  (time, lat, lon) float32 27MB ...
    precip_mean    (time) float32 31kB ...
    crs            int32 4B ...
Attributes:
    title:        GPM IMERG Late Run — San Joaquin Valley subset
    source:       NASA GPM IMERG Late Run V06
    shapefile:    C:\Users\mwwan\Documents\Research_Data\sjvw_boundaries\smal...
    date_range:   2000-01-01 to 2020-12-31
    created:      2026-05-20T18:43:04Z
    Conventions:  CF-1.8

In [ ]:
import numpy as np
import geopandas as gpd
import contextily as ctx

import matplotlib.pyplot as plt

da_t = db.sel(time='2010-01-01').precipitation  # (lat, lon) DataArray

# build point GeoDataFrame
lons, lats = np.meshgrid(da_t.lon.values, da_t.lat.values)
vals = da_t.values.ravel()
gdf = gpd.GeoDataFrame({'precip': vals},
                       geometry=gpd.points_from_xy(lons.ravel(), lats.ravel()),
                       crs="EPSG:4326")
gdf = gdf[~np.isnan(gdf['precip'])]

# plot (use web-mercator basemap if contextily is available)
try:
    ax = gdf.to_crs(epsg=3857).plot(column='precip', cmap='Blues', markersize=8, legend=True)
    ctx.add_basemap(ax, source=ctx.providers.Stamen.TerrainBackground)
else:
    ax = gdf.plot(column='precip', cmap='Blues', markersize=8, legend=True)

ax.set_title("Precipitation 2010-01-01")
ax.set_axis_off()
plt.show()

<xarray.DataArray 'precipitation' (lat: 38, lon: 35)> Size: 5kB
[1330 values with dtype=float32]
Coordinates:
  * lat      (lat) float32 152B 34.85 34.95 35.05 35.15 ... 38.35 38.45 38.55
  * lon      (lon) float32 140B -121.9 -121.8 -121.8 ... -118.8 -118.7 -118.6
    time     datetime64[ns] 8B 2010-01-01
Attributes:
    units:         mm/day
    long_name:     daily precipitation rate (basin-masked)
    grid_mapping:  crs